# Preprocessing & Clustering Preparation

In this notebook, I will prepare the restaurant-level dataset for unsupervised machine learning.

I will handle missing values, transform categorical variables into numerical representations, and scale numerical features to ensure that the variables are suitable for clustering algorithms.

The objective is to construct a clean, numerical feature matrix that captures meaningful similarities and differences between restaurants while avoiding unnecessary features and distortions.

## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Path Definition
project_root = Path.cwd().parent
data_path = project_root / "1. data"
processed_data_path = data_path / "processed" 

In [3]:
# Loading data
data_file = processed_data_path / "restaurant_features.csv" 
restaurant_df = pd.read_csv(data_file)
display(restaurant_df.head())

,Restaurant,Cost,Collections,Cuisines,avg_rating,rating_std,median_rl,avg_pic
0,Beyond Flavours,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...",4.28,1.239746,32.0,0.39
1,Paradise,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese",4.70,0.771984,28.0,0.02
2,Flechazo,"1,300","Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts",4.66,0.878704,30.5,0.55
3,Shah Ghouse Hotel & Restaurant,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever...",3.21,1.603626,32.5,0.30
4,Over The Moon Brew Company,"1,200","Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med...",4.34,1.065719,36.0,0.44


## 2. Missing Value Assessment

The merged dataset contains missing values in the review-derived features for restaurants without available review information.

Since these missing values represent unavailable customer feedback rather than actual restaurant characteristics, I will examine the affected restaurants before deciding how to handle them.

In [4]:
# Identification of missing features
review_columns = [
    "avg_rating",
    "rating_std",
    "median_rl",
    "avg_pic",
]
missing_review_data = restaurant_df.loc[
    restaurant_df[review_columns].isna().any(axis=1),
    ["Restaurant"] + review_columns
]
display(missing_review_data)

,Restaurant,avg_rating,rating_std,median_rl,avg_pic
100,IndiBlaze,NaN,NaN,NaN,NaN
101,Sweet Basket,NaN,NaN,NaN,NaN
102,Angaara Counts 3,NaN,NaN,NaN,NaN
103,Wich Please,NaN,NaN,NaN,NaN
104,Republic Of Noodles - Lemon Tree Hotel,NaN,NaN,NaN,NaN


## 3. Categorical Feature Processing

The dataset contains categorical features that need to be converted into numerical representations before they can be used by clustering algorithms.

Since some categorical features contain multiple labels per restaurant, I will use appropriate encoding techniques to represent them while avoiding unnecessary complexity.

### 3.1 Cuisine Encoding

The Cuisines column contains multiple cuisine labels separated by commas. Since a restaurant can offer several cuisines simultaneously, I will use multi-label binary encoding to represent each individual cuisine as a separate feature.

This approach allows restaurants to share similarities based on the cuisines they offer, rather than treating every unique combination of cuisines as a separate category.

In [5]:
# Splitting the Cuisine Labels
cuisine_list = restaurant_df["Cuisines"].str.split(",").apply(
    lambda cuisines : [cuisine.strip() for cuisine in cuisines]
)
display(cuisine_list.head())

0    [Chinese, Continental, Kebab, European, South ...
1                     [Biryani, North Indian, Chinese]
2       [Asian, Mediterranean, North Indian, Desserts]
3    [Biryani, North Indian, Chinese, Seafood, Beve...
4    [Asian, Continental, North Indian, Chinese, Me...
Name: Cuisines, dtype: object

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

cuisine_encoder = MultiLabelBinarizer()         # Encoding Labels
cuisine_encoded = cuisine_encoder.fit_transform(cuisine_list)

# Dataframe of Cuisine Features
cuisine_features = pd.DataFrame(
    cuisine_encoded,
    columns=[f"cuisine_{cuisine}" for cuisine in cuisine_encoder.classes_],
    index= restaurant_df.index
)
display(cuisine_features.head())
print("Number of Cuisine Features: ", cuisine_features.shape[1])

,cuisine_American,cuisine_Andhra,cuisine_Arabian,cuisine_Asian,cuisine_BBQ,cuisine_Bakery,cuisine_Beverages,cuisine_Biryani,cuisine_Burger,cuisine_Cafe,...,cuisine_North Indian,cuisine_Pizza,cuisine_Salad,cuisine_Seafood,cuisine_South Indian,cuisine_Spanish,cuisine_Street Food,cuisine_Sushi,cuisine_Thai,cuisine_Wraps
0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,1,0,0,...,1,0,0,1,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


Number of Cuisine Features:  44


### 3.2 Collections

The Collections feature represents curated groups assigned to restaurants by Zomato, such as *Best Bars & Pubs*, *Great Buffets*, and *Late Night Restaurants*.

Although these collections may provide useful context, the feature contains a substantial number of missing values and multiple distinct categories. Encoding it would introduce additional sparse features into an already relatively small dataset.

To avoid increasing dimensionality unnecessarily and relying heavily on platform-specific categorizations, I will exclude Collections from the clustering feature matrix.

However, I will retain the original column in the dataset for use during cluster analysis, restaurant profiling, and visualization.

## 4. Numerical Feature Processing

The dataset contains numerical features representing restaurant pricing, customer ratings, review characteristics, and customer engagement.

Before preparing these features for clustering, I will address missing values in the review-derived features and ensure that all numerical variables are represented appropriately.

Since clustering algorithms based on distance are sensitive to differences in feature scales, I will also standardize the numerical features so that variables with larger magnitudes do not disproportionately influence the clustering results.

In [7]:
# Restaunrant with missing Review Data
restaurant_df["has_review_data"] = restaurant_df["avg_rating"].notna().astype(int)
restaurant_df["has_review_data"].value_counts()

has_review_data
1    100
0      5
Name: count, dtype: int64

In [8]:
# Imputing missing features
review_columns= [
    "avg_rating",
    "rating_std",
    "median_rl",
    "avg_pic"
]
for column in review_columns:
    restaurant_df[column] = restaurant_df[column].fillna(
        restaurant_df[column].median()
    )
restaurant_df[review_columns].isna().sum()

avg_rating    0
rating_std    0
median_rl     0
avg_pic       0
dtype: int64

In [9]:
# Inspecting Feature Statistics
numerical_column = [
    "Cost",
    "avg_rating", "rating_std",
    "median_rl", "avg_pic" 
]
restaurant_df[numerical_column].describe()


,avg_rating,rating_std,median_rl,avg_pic
count,105.000000,105.000000,105.000000,105.000000
mean,3.603174,1.380770,33.757143,0.750527
std,0.495531,0.248154,14.804351,0.720677
min,2.400000,0.498077,2.500000,0.010000
25%,3.260000,1.250010,28.000000,0.170000
50%,3.613081,1.429946,35.000000,0.560000
75%,3.845000,1.546452,43.000000,1.050000
max,4.880000,1.853716,62.500000,3.560000


In [10]:
# Inspecting Cost
restaurant_df["Cost"].describe()

count     105
unique     29
top       500
freq       13
Name: Cost, dtype: object

In [11]:
# Inspecting Cost Values
restaurant_df["Cost"].unique()

<ArrowStringArray>
[  '800', '1,300', '1,200', '1,500',   '500',   '300', '1,000',   '350',
   '400', '1,600',   '750',   '550', '1,900',   '450',   '150', '1,400',
 '1,100',   '600',   '200',   '900',   '700', '1,700', '2,500',   '850',
   '650', '1,800', '2,800', '1,750',   '250']
Length: 29, dtype: str

In [12]:
# Conversion of Cost to Numeric
restaurant_df["Cost"] = pd.to_numeric(
    restaurant_df["Cost"].str.replace(",","", regex=False)
)
restaurant_df["Cost"].dtype         # Checking

dtype('int64')

In [13]:
# Inspecting All Feature Statistics
numerical_column = [
    "Cost",
    "avg_rating", "rating_std",
    "median_rl", "avg_pic" 
]
restaurant_df[numerical_column].describe()


,Cost,avg_rating,rating_std,median_rl,avg_pic
count,105.000000,105.000000,105.000000,105.000000,105.000000
mean,861.428571,3.603174,1.380770,33.757143,0.750527
std,510.149730,0.495531,0.248154,14.804351,0.720677
min,150.000000,2.400000,0.498077,2.500000,0.010000
25%,500.000000,3.260000,1.250010,28.000000,0.170000
50%,700.000000,3.613081,1.429946,35.000000,0.560000
75%,1200.000000,3.845000,1.546452,43.000000,1.050000
max,2800.000000,4.880000,1.853716,62.500000,3.560000


### 4.1 Handling the Cost Feature

During the inspection of numerical features, I discovered that the Cost column was stored as a string rather than a numerical data type. Consequently, its initial summary returned categorical statistics instead of numerical ones.

Upon inspecting the unique values, I found that the cost entries contained commas as thousands separators (e.g., 1,300 and 2,500), which prevented them from being directly interpreted as numerical values.

To address this, I removed the commas and converted the column to an integer data type. This ensures that Cost is treated as a numerical feature and can be properly included in subsequent scaling and clustering operations.

After correcting the data type, I re-examined the numerical features to confirm that all five features were represented numerically and to understand their respective ranges and distributions before scaling.

### 4.2 Scaling Numerical Features

The numerical features in our dataset have different ranges and units. For example, Cost ranges from 150 to 2800, while avg_rating ranges from 2.4 to 4.88. Without scaling, features with larger numerical values could disproportionately influence distance-based clustering algorithms.

To address this, I will scale the numerical features so that they are on comparable scales.

I will use RobustScaler, which scales features using their median and interquartile range (IQR) instead of the mean and standard deviation. This makes it less sensitive to extreme values and is suitable for features that may contain outliers.

Scaling will help prevent features with larger magnitudes from dominating the clustering process.

In [14]:
from sklearn.preprocessing import RobustScaler

# Scaling Features
scaler = RobustScaler()
numerical_scaled = scaler.fit_transform(
    restaurant_df[numerical_column]
)
numerical_features = pd.DataFrame(
    numerical_scaled, columns= numerical_column, 
    index=restaurant_df.index
)
display(numerical_features.head())

,Cost,avg_rating,rating_std,median_rl,avg_pic
0,0.142857,1.140033,-0.641610,-0.200000,-0.193182
1,0.142857,1.857982,-2.219528,-0.466667,-0.613636
2,0.857143,1.789605,-1.859525,-0.300000,-0.011364
3,0.142857,-0.689027,0.585880,-0.166667,-0.295455
4,0.714286,1.242597,-1.228660,0.066667,-0.136364


In [15]:
# Validation of Features
numerical_features.describe()

,Cost,avg_rating,rating_std,median_rl,avg_pic
count,105.000000,105.000000,105.000000,105.000000,105.000000
mean,0.230612,-0.016935,-0.165890,-0.082857,0.216507
std,0.728785,0.847062,0.837107,0.986957,0.818951
min,-0.785714,-2.073642,-3.143509,-2.166667,-0.625000
25%,-0.285714,-0.603557,-0.606985,-0.466667,-0.443182
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.714286,0.396443,0.393015,0.533333,0.556818
max,3.000000,2.165674,1.429518,1.833333,3.409091


## 5. Final Feature Matrix

After preprocessing the numerical and categorical features, I will combine them into a single feature matrix for clustering.

The numerical features have been imputed where necessary and scaled using RobustScaler, while the cuisine features have been encoded into binary representations using MultiLabelBinarizer.

I will combine these two sets of features while excluding the restaurant names and other non-feature columns. This ensures that the clustering algorithms operate only on the prepared numerical feature representations.

The original dataset will be retained separately for reference and interpretation.

In [16]:
# Combining Features
clustering_features = pd.concat(
    [numerical_features, cuisine_features], axis=1
)
display(clustering_features.head())
print("Feature Matrix Shape: ", clustering_features.shape)

,Cost,avg_rating,rating_std,median_rl,avg_pic,cuisine_American,cuisine_Andhra,cuisine_Arabian,cuisine_Asian,cuisine_BBQ,...,cuisine_North Indian,cuisine_Pizza,cuisine_Salad,cuisine_Seafood,cuisine_South Indian,cuisine_Spanish,cuisine_Street Food,cuisine_Sushi,cuisine_Thai,cuisine_Wraps
0,0.142857,1.140033,-0.641610,-0.200000,-0.193182,0,0,0,0,0,...,1,0,0,0,1,0,0,0,0,0
1,0.142857,1.857982,-2.219528,-0.466667,-0.613636,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0.857143,1.789605,-1.859525,-0.300000,-0.011364,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0.142857,-0.689027,0.585880,-0.166667,-0.295455,0,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
4,0.714286,1.242597,-1.228660,0.066667,-0.136364,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0


Feature Matrix Shape:  (105, 49)


In [17]:
# Final Validation of Features
print("Feature Matrix Shape: ", clustering_features.shape)
print("Missing Values: ", clustering_features.isna().sum().sum())
print("Non-numeric Columns: ", clustering_features.select_dtypes(exclude="number").columns.tolist())
print("Duplicate Feature Names: ", clustering_features.columns.duplicated().sum())
print("Cuisine Values: ", sorted(cuisine_features.stack().unique()))

Feature Matrix Shape:  (105, 49)
Missing Values:  0
Non-numeric Columns:  []
Duplicate Feature Names:  0
Cuisine Values:  [np.int64(0), np.int64(1)]


### 5.1 Saving the Prepared Features

After completing the preprocessing steps, I have obtained a numerical feature matrix containing 105 restaurants and 49 features.

The feature matrix contains the scaled numerical features and binary encoded cuisine features, with no missing values or non-numeric columns.

I will save the prepared feature matrix along with the fitted preprocessing objects so they can be reused in subsequent stages of the project without repeating the transformations.

In [18]:
import joblib

# Saving the Prepared Matrix
clustering_features.to_csv(
    processed_data_path / "clustering_features.csv", 
    index=False
)
# Saving the preprocessing objects
models_path = project_root / "4. models"
models_path.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, models_path / "numerical_scaler.pkl")
joblib.dump(cuisine_encoder, models_path / "cuisine_encoder.pkl")
print("Feature Matrix is successfully saved.")
print("Preprocessing Objects saved successfully")

Feature Matrix is successfully saved.
Preprocessing Objects saved successfully
